# Simulating a nonlinear system: NumPy / SciPy, minilink

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/mass_spring_nonlinear_simulation.ipynb)

A mass–spring–damper whose damper is strongly nonlinear (damping force proportional to $\dot x^3$):

$$m\ddot{x} = u - k x - b\dot{x}^3$$

The same three steps, twice, with **NumPy / SciPy / Matplotlib** only, then with **minilink**:

1. **Equations of motion in state-space form**: $\mathbf{x} = [x, \dot x]$, $\dot{\mathbf{x}} = f(\mathbf{x}, u)$.
2. **Open-loop simulation**: an input signal $u(t)$ drives the plant, here a constant force $u = 1$ from rest.
3. **Closed-loop simulation**: a reference signal $r(t)$ and a control law $u = g(\mathbf{x}, r)$, here $u = k_p (r - x) - k_d \dot x$.

Each part is written as a template: replace $f$ with your own equations of motion, $u(t)$ and $r(t)$ with your own signals, and $g$ with your own control law.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## Part 1 — NumPy, SciPy and Matplotlib

Simulating means integrating $\dot{\mathbf x} = f(\mathbf x, u)$ over time: `solve_ivp` does it, given a Python function that returns $\dot{\mathbf x}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

### Equations of motion

$\dot{\mathbf x} = f(\mathbf x, u) = \begin{bmatrix} \dot x \\ (u - k x - b \dot x^3)/m \end{bmatrix}$

In [ ]:
m, k, b = 1.0, 1.0, 1.0


def f(x, u):
    """Equations of motion: state x = [x, ẋ], input u = force (N), returns [ẋ, ẍ]."""
    pos, vel = x

    # m ẍ = u − k x − b ẋ³
    acc = (u - k * pos - b * vel**3) / m

    return np.array([vel, acc])

### Open loop

The input signal is a function of time, $u(t)$. `solve_ivp` expects a function $(t, \mathbf x) \mapsto \dot{\mathbf x}$: we write it explicitly by combining the input signal and the equations of motion.

In [ ]:
def input_signal(t):
    """u(t): a constant force of 1 N."""
    u = 1.0

    return u


def f_open_loop(t, x):
    """ẋ = f(x, u(t))"""
    u = input_signal(t)

    dx = f(x, u)

    return dx


x0 = np.array([0.0, 0.0])
t = np.linspace(0.0, 30.0, 3001)

sol = solve_ivp(f_open_loop, [0.0, 30.0], x0, t_eval=t)

plt.plot(sol.t, sol.y[0])
plt.xlabel("t [s]")
plt.ylabel("x [m]")
plt.grid(True)
plt.show()

### Closed loop

The reference is a function of time, $r(t)$, and the control law a function of the state and of the reference, $u = g(\mathbf x, r)$. Plugging both into $f$ gives the closed-loop dynamics, integrated the same way.

In [ ]:
kp, kd = 10.0, 5.0


def reference_signal(t):
    """r(t): a constant reference of 1 m."""
    r = 1.0

    return r


def control_law(x, r):
    """u = g(x, r): state feedback kp (r − x) − kd ẋ."""
    pos, vel = x

    u = kp * (r - pos) - kd * vel

    return u


def f_closed_loop(t, x):
    """ẋ = f(x, g(x, r(t)))"""
    r = reference_signal(t)
    u = control_law(x, r)

    dx = f(x, u)

    return dx


t = np.linspace(0.0, 10.0, 1001)

sol = solve_ivp(f_closed_loop, [0.0, 10.0], x0, t_eval=t)
u = [control_law(x, reference_signal(tk)) for tk, x in zip(sol.t, sol.y.T)]

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(sol.t, sol.y[0])
ax[1].plot(sol.t, u)
ax[0].set_ylabel("x [m]")
ax[1].set_ylabel("u [N]")
ax[1].set_xlabel("t [s]")
ax[0].grid(True)
ax[1].grid(True)
plt.show()

The position settles at $x = \dfrac{k_p}{k_p + k}\,r = 0.91$: a control law without integral action leaves a static error, since holding the spring stretched takes a nonzero $k_p (r - x)$.

## Part 2 — minilink

In [minilink](https://github.com/alx87grd/minilink), every piece is a block: the plant is a `DynamicSystem` with its equations `f` and `h`, a signal is a `System` whose output depends on `t`, and the controller is a `Controller` with its `control_law`. Blocks are wired with `>>` (series) and `@` (feedback); `compute_trajectory` and `plot_trajectory` do the rest. minilink calls every method as `(x, u, t, params)`; the arguments we do not need stay unused.

In [ ]:
import numpy as np
from minilink import Controller, DynamicSystem, System

### Equations of motion

`f` is the state equation and `h` the output equation: here the whole state is measured, $\mathbf y = \mathbf x$ (minilink's default when `output_dim = n`; override `h` for another sensor).

In [ ]:
m, k, b = 1.0, 1.0, 1.0


class NonlinearMassSpringDamper(DynamicSystem):
    """State x = [x, ẋ] (m, m/s), input u = [force] (N), output y = x."""

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)

    def f(self, x, u, t=0, params=None):
        pos, vel = x
        force = u[0]

        # m ẍ = u − k x − b ẋ³
        acc = (force - k * pos - b * vel**3) / m

        return np.array([vel, acc])

    def h(self, x, u, t=0, params=None):
        y = x

        return y


plant = NonlinearMassSpringDamper()

### Open loop

The input signal is a source block: no state, one output $u(t)$. `>>` connects it to the plant.

In [ ]:
class InputSignal(System):
    """u(t): a constant force of 1 N."""

    def __init__(self):
        super().__init__()
        self.id = "source"
        self.add_output_port("u", function=self.signal)

    def signal(self, x, u, t=0, params=None):
        force = 1.0

        return force


open_loop = InputSignal() >> plant

open_loop.plot_diagram()

In [ ]:
open_loop.compute_trajectory(tf=30.0, verbose=False)

open_loop.plot_trajectory()

### Closed loop

The controller is a block with two inputs, the measurement $\mathbf y = [x, \dot x]$ and the reference $r$, and one output $u$; `control_law` receives the inputs concatenated in port order, and `dependencies="all"` declares that $u$ depends directly on them (without it, minilink evaluates the block with zero inputs). `@` connects the plant output `y` to the controller input `y` and the controller output `u` to the plant input `u`: the port names do the wiring. The reference is another source block, connected with `>>`.

In [ ]:
kp, kd = 10.0, 5.0


class StateFeedback(Controller):
    """u = g(y, r) = kp (r − x) − kd ẋ: inputs y = [x, ẋ] and r, output u = force (N)."""

    def __init__(self):
        super().__init__()
        self.add_input_port("y", dim=2)
        self.add_input_port("r")
        self.add_output_port("u", function=self.control_law, dependencies="all")

    def control_law(self, x, u, t=0, params=None):
        pos, vel, r = u  # the block inputs, in port order

        force = kp * (r - pos) - kd * vel

        return force


class ReferenceSignal(System):
    """r(t): a constant reference of 1 m."""

    def __init__(self):
        super().__init__()
        self.add_output_port("r", function=self.signal)

    def signal(self, x, u, t=0, params=None):
        r = 1.0

        return r


closed_loop = ReferenceSignal() >> (StateFeedback() @ plant)

closed_loop.plot_diagram()

In [ ]:
closed_loop.compute_trajectory(tf=10.0, verbose=False)

closed_loop.plot_trajectory()

### Bonus: the phase plane

The open-loop trajectory in the $(x, \dot x)$ plane, overlaid on the vector field $f(\mathbf x, u = 1)$.

In [ ]:
plant.plot_phase_plane(open_loop.traj, u=[1.0])